# TickNet L Midterm Final Kaggle

Run All notebook cho dữ liệu đã chuẩn hóa sẵn: `Mid32/train|test/<5 classes>` và `Mid224/train|test/<5 classes>`. Không chia lại hay copy ảnh. Notebook chạy **TickNet-L v1** (`--model l`) trên cả hai dataset, lưu log theo epoch và xuất hình/bảng cho báo cáo.

Trước khi Run All: chọn GPU, bật Internet, Add Input hai dataset `midd-32` và `midd-224`.

In [ ]:
# Clone đúng nhánh giữa kỳ.
!test -d /kaggle/working/TickNets/.git || git clone --depth 1 --branch feature/giua-ki-model-l https://github.com/khuonglaptri3/TickNets.git /kaggle/working/TickNets
%cd /kaggle/working/TickNets
!nvidia-smi


In [ ]:
from pathlib import Path
from datetime import datetime
import csv, hashlib, json, os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision

WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
CLASSES = ('bird', 'cat', 'dog', 'frog', 'horse')
RUN_TAG = datetime.now().strftime('seed42_%Y%m%d_%H%M%S')
RUNS = WORK / 'runs'
REPORT = WORK / 'midterm_report_assets'
DATA_ROOT = WORK / 'midterm_data'
RUNS.mkdir(exist_ok=True)
REPORT.mkdir(exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('Bật GPU tại Notebook Settings rồi Run All lại.')
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('run:', RUN_TAG)


## Locate and validate the attached datasets

Notebook tự tìm đúng thư mục `Mid32` và `Mid224` trong Kaggle Input. Nó yêu cầu mỗi variant có đủ `train/test` và năm lớp, 5.000 ảnh train + 50 ảnh test trên mỗi lớp.

In [ ]:
def is_variant(path: Path):
    return (path / 'train').is_dir() and (path / 'test').is_dir() and all(
        (path / split / cls).is_dir() for split in ('train', 'test') for cls in CLASSES
    )

def locate_variant(variant: str) -> Path:
    direct = [p / variant for p in INPUT.iterdir() if p.is_dir()]
    matches = [p for p in direct if is_variant(p)]
    if not matches:
        # Fallback if Kaggle added an extra directory level.
        matches = [p for p in INPUT.rglob(variant) if p.is_dir() and is_variant(p)]
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one valid {variant}; found: {matches}')
    return matches[0]

RAW = {'Mid32': locate_variant('Mid32'), 'Mid224': locate_variant('Mid224')}
rows = []
for variant, root in RAW.items():
    print(f'\n{variant}: {root}')
    for split, expected in (('train', 5000), ('test', 50)):
        counts = {cls: len(list((root / split / cls).glob('*.jpeg'))) for cls in CLASSES}
        print(split, counts)
        if set(counts.values()) != {expected}:
            raise ValueError(f'{variant}/{split} must have {expected} JPEGs in every class: {counts}')
        for cls, n in counts.items():
            rows.append({'variant': variant, 'split': split, 'class': cls, 'images': n})

# Create only two symlinks and a small provenance manifest; images remain Kaggle Inputs.
DATA_ROOT.mkdir(exist_ok=True)
for variant, source in RAW.items():
    link = DATA_ROOT / variant
    if not (link.exists() or link.is_symlink()):
        link.symlink_to(source, target_is_directory=True)
    elif link.resolve() != source.resolve():
        raise RuntimeError(f'{link} points to a different dataset. Restart Session.')

manifest = DATA_ROOT / 'split_manifest.csv'
with manifest.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=('variant', 'split', 'class_name', 'filename'))
    writer.writeheader()
    for variant, root in RAW.items():
        for split in ('train', 'test'):
            for cls in CLASSES:
                for image in sorted((root / split / cls).glob('*.jpeg')):
                    writer.writerow({'variant': variant, 'split': split, 'class_name': cls, 'filename': image.name})

dataset_table = pd.DataFrame(rows)
display(dataset_table)
dataset_table.to_csv(REPORT / 'dataset_properties_by_class.csv', index=False)
print('Dataset validation passed. Manifest SHA-256:', hashlib.sha256(manifest.read_bytes()).hexdigest())


In [ ]:
%cd /kaggle/working/TickNets
!python train_mid.py --data-root /kaggle/working/midterm_data --variant Mid32 --seed 42 --check-data
!python train_mid.py --data-root /kaggle/working/midterm_data --variant Mid224 --seed 42 --check-data
!python profile_mid.py --models l --output /kaggle/working/midterm_report_assets/model_profiles_l.json


## Official training — TickNet L

Hai cell sau chạy tuần tự khi chọn **Run All**. Mỗi run là 200 epoch, SGD + momentum 0.9 + CosineAnnealingLR. Nếu Mid224 báo CUDA out-of-memory, dừng run và đổi batch size từ 64 sang 32.

In [ ]:
MID32_RUN = RUNS / f'l_mid32_{RUN_TAG}'
!python train_mid.py --data-root /kaggle/working/midterm_data --variant Mid32 --model l --seed 42 --epochs 200 --batch-size 128 --num-workers 2 --threads 2 --device cuda:0 --output-dir {MID32_RUN}


In [ ]:
MID224_RUN = RUNS / f'l_mid224_{RUN_TAG}'
!python train_mid.py --data-root /kaggle/working/midterm_data --variant Mid224 --model l --seed 42 --epochs 200 --batch-size 64 --num-workers 2 --threads 2 --device cuda:0 --output-dir {MID224_RUN}


## Export report evidence

The original pipeline tests only at the final checkpoint. This cell creates detailed final-test metrics, per-class Precision/Recall/F1, confusion matrices, and learning curves without using test data to select a model.

In [ ]:
from models.mid_data import build_mid_loaders, seed_everything
from models.mid_models import build_mid_model
from sklearn.metrics import classification_report, confusion_matrix

def export_evaluation(variant, run):
    checkpoint = torch.load(run / 'last.pt', map_location='cpu', weights_only=True)
    config = checkpoint['config']
    seed_everything(config['seed'])
    _, loader = build_mid_loaders(DATA_ROOT, variant, batch_size=64, seed=config['seed'], num_workers=2, augment=False, pin_memory=True)
    model = build_mid_model('l', num_classes=5, variant=variant).cuda().eval()
    model.load_state_dict(checkpoint['model_state_dict'])
    y_true, y_pred = [], []
    with torch.inference_mode():
        for images, labels in loader:
            y_pred.extend(model(images.cuda(non_blocking=True)).argmax(1).cpu().tolist())
            y_true.extend(labels.tolist())
    cm = confusion_matrix(y_true, y_pred, labels=range(5))
    report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
    prefix = variant.lower()
    pd.DataFrame(report).T.to_csv(REPORT / f'{prefix}_classification_report.csv')
    np.savetxt(REPORT / f'{prefix}_confusion_matrix.csv', cm, fmt='%d', delimiter=',')
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues'); fig.colorbar(im, ax=ax)
    ax.set(xticks=range(5), yticks=range(5), xticklabels=CLASSES, yticklabels=CLASSES, xlabel='Predicted', ylabel='True', title=f'TickNet-L — {variant}')
    for i in range(5):
        for j in range(5): ax.text(j, i, cm[i,j], ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black')
    fig.tight_layout(); fig.savefig(REPORT / f'{prefix}_confusion_matrix.png', dpi=180); plt.show()
    return {'variant': variant, 'test_top1_percent': round(100*np.mean(np.array(y_true)==np.array(y_pred)), 4), 'macro_f1': report['macro avg']['f1-score']}

def export_curves(variant, run):
    h = pd.read_csv(run / 'epochs.csv')
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h.epoch, h.train_loss); ax[0].set(title=f'{variant} train loss', xlabel='Epoch', ylabel='Loss')
    ax[1].plot(h.epoch, h.train_top1); ax[1].set(title=f'{variant} train Top-1', xlabel='Epoch', ylabel='Accuracy (%)')
    for a in ax: a.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(REPORT / f'{variant.lower()}_learning_curves.png', dpi=180); plt.show()

results = [export_evaluation('Mid32', MID32_RUN), export_evaluation('Mid224', MID224_RUN)]
for variant, run in [('Mid32', MID32_RUN), ('Mid224', MID224_RUN)]: export_curves(variant, run)
summary = pd.DataFrame(results)
for variant, run in [('Mid32', MID32_RUN), ('Mid224', MID224_RUN)]:
    cfg = json.loads((run / 'config.json').read_text()); test = json.loads((run / 'test_metrics.json').read_text())
    summary.loc[summary.variant == variant, ['parameters','flops','epochs','batch_size','learning_rate','momentum','weight_decay','final_test_loss']] = [cfg['learnable_parameters'],cfg['complexity']['flops'],cfg['epochs'],cfg['batch_size'],cfg['learning_rate'],cfg['momentum'],cfg['weight_decay'],test['loss']]
display(summary)
summary.to_csv(REPORT / 'midterm_required_summary.csv', index=False)


In [ ]:
# Download this file from Kaggle Output after Save Version.
!cd /kaggle/working && zip -r ticknet_l_midterm_final_artifacts.zip runs midterm_report_assets
print('Ready: /kaggle/working/ticknet_l_midterm_final_artifacts.zip')
